# 01 — EDA & Preprocessing: Pittsburgh Tobacco Retail Inspections

**Project:** Optimizing the Order and Execution of Tobacco Retail Inspections in Pittsburgh, PA  
**Purpose:** Audit the raw FDA export, establish observation grain, resolve retailer/location identifiers conservatively, validate the underage-sale outcome, perform temporal EDA, and export reproducible processed tables for later enrichment and modeling.

> **Analytical rule:** The original `Pittsburgh Inspections.csv` is the source of truth. Previously cleaned/master files are used only as audit comparators. No optimization is performed here.

### Key distinction used throughout
- **FDA record:** one row in the source export. We do not assume this is identical to a unique physical inspection until tested.
- **Location:** normalized street address + ZIP (`location_key`), useful for geocoding/routing.
- **Retailer entity (conservative):** normalized retailer name + location (`retailer_entity_key`). This avoids automatically merging different businesses that share an address. Name changes/ownership changes remain an entity-resolution issue to review.

## 1. Setup and imports
Paths are centralized here. The notebook can run from the repository root or from a directory containing the supplied files.

In [ ]:
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path.cwd()
RAW_FILE = ROOT / "Pittsburgh Inspections.csv"
PRIOR_CLEAN_FILE = ROOT / "pittsburgh_inspections_cleaned.csv"
PRIOR_MASTER_FILE = ROOT / "pittsburgh_retailer_master_stage1.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUT_CLEAN = PROCESSED_DIR / "pittsburgh_inspections_cleaned.csv"
OUT_MASTER = PROCESSED_DIR / "pittsburgh_retailer_master.csv"
OUT_QUALITY = PROCESSED_DIR / "data_quality_summary.csv"

for p in [RAW_FILE, PRIOR_CLEAN_FILE, PRIOR_MASTER_FILE]:
    print(f"{p.name}: {'FOUND' if p.exists() else 'not found'}")
assert RAW_FILE.exists(), f"Missing required raw file: {RAW_FILE}"

## 2. Load raw data
We inspect the physical file first so the FDA preamble is visible rather than silently skipped.

In [ ]:
with RAW_FILE.open("r", encoding="utf-8-sig", errors="replace") as f:
    first_lines = [next(f).rstrip("\n") for _ in range(10)]
for i, line in enumerate(first_lines, start=1):
    print(f"{i:02d}: {line}")

In [ ]:
# The actual CSV header is on physical line 8 in this export, so seven preamble rows are skipped.
raw = pd.read_csv(RAW_FILE, skiprows=7, dtype=str, keep_default_na=True)
print(f"Loaded {raw.shape[0]:,} rows × {raw.shape[1]} columns")
display(raw.head())

## 3. Raw-data audit
No recoding or row deletion occurs before this audit. Placeholder strings are counted separately from true nulls.

In [ ]:
EXPECTED_COLUMNS = [
    "Retailer Name", "Street Address", "City", "State", "Zip", "UP Involved",
    "Sale to UP", "Product Type", "Brand", "Flavor", "Inspection Date",
    "Decision Date", "Outcome", "Link", "Charges", "Unauthorized Product Charge",
    "Nicotine Source"
]
assert raw.columns.tolist() == EXPECTED_COLUMNS, "Unexpected source schema; inspect before proceeding."

raw_audit = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "true_missing_n": raw.isna().sum(),
    "true_missing_pct": (100 * raw.isna().mean()).round(2),
    "unique_nonmissing": raw.nunique(dropna=True),
})
display(raw_audit)
print("Exact duplicate rows:", int(raw.duplicated().sum()))
print("Unique retailer names:", raw["Retailer Name"].nunique(dropna=True))
print("Unique raw street address + ZIP pairs:", raw[["Street Address", "Zip"]].drop_duplicates().shape[0])

In [ ]:
important_categoricals = [
    "UP Involved", "Sale to UP", "Product Type", "Brand", "Flavor",
    "Outcome", "Unauthorized Product Charge", "Nicotine Source"
]
for col in important_categoricals:
    print(f"\n--- {col} ---")
    display(raw[col].value_counts(dropna=False).head(25).to_frame("count"))

## 4. Missingness and data-quality checks
Placeholder strings are not treated as genuine values. We first inventory them, then create analysis copies where documented placeholders become `NaN`.

In [ ]:
PLACEHOLDER_PATTERN = r"^(?:N/?A|NA|NOT AVAILABLE|NONE|NULL|UNKNOWN|--)?$"
placeholder_counts = {}
for col in raw.columns:
    s = raw[col].astype("string").str.strip().str.upper()
    placeholder_counts[col] = int(s.str.match(PLACEHOLDER_PATTERN, na=False).sum())
placeholder_table = pd.DataFrame.from_dict(placeholder_counts, orient="index", columns=["placeholder_string_n"])
display(raw_audit.join(placeholder_table))

In [ ]:
# Parse dates on temporary series for auditing; original string columns remain untouched.
inspection_date_audit = pd.to_datetime(raw["Inspection Date"], errors="coerce")
decision_date_audit = pd.to_datetime(raw["Decision Date"], errors="coerce")

date_audit = pd.DataFrame({
    "field": ["Inspection Date", "Decision Date"],
    "parseable_n": [inspection_date_audit.notna().sum(), decision_date_audit.notna().sum()],
    "unparseable_or_missing_n": [inspection_date_audit.isna().sum(), decision_date_audit.isna().sum()],
    "min_date": [inspection_date_audit.min(), decision_date_audit.min()],
    "max_date": [inspection_date_audit.max(), decision_date_audit.max()],
})
display(date_audit)

both_dates = inspection_date_audit.notna() & decision_date_audit.notna()
impossible_order = both_dates & (decision_date_audit < inspection_date_audit)
print("Rows with decision date before inspection date:", int(impossible_order.sum()))
if impossible_order.any():
    display(raw.loc[impossible_order, ["Retailer Name", "Street Address", "Inspection Date", "Decision Date", "Outcome"]].head(20))

### Date interpretation
`Inspection Date` is missing/unavailable for much of this export, while `Decision Date` is substantially more complete. Therefore, this notebook **does not impute inspection dates from decision dates**. It preserves both fields and uses `Decision Date` as an explicitly labeled proxy for broad temporal EDA when inspection dates are unavailable. A later predictive model must define its prediction timestamp carefully.

## 5. Cleaning and standardization
Cleaning is deliberately conservative. We normalize case/whitespace and ZIP formatting, but do not perform aggressive street-suffix substitutions that could collapse distinct source strings without review.

In [ ]:
def normalize_text(series):
    return (series.astype("string")
                  .str.strip()
                  .str.upper()
                  .str.replace(r"\s+", " ", regex=True))

def normalize_address(series):
    # Conservative: whitespace/case/punctuation only. No suffix substitutions.
    return (normalize_text(series)
            .str.replace(r"[.,]", "", regex=True)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip())

clean = raw.copy()
clean["retailer_name_normalized"] = normalize_text(clean["Retailer Name"])
clean["street_address_normalized"] = normalize_address(clean["Street Address"])
clean["city_normalized"] = normalize_text(clean["City"])
clean["state_normalized"] = normalize_text(clean["State"])
clean["zip5"] = clean["Zip"].astype("string").str.extract(r"(\d{5})", expand=False)
clean["inspection_date_parsed"] = pd.to_datetime(clean["Inspection Date"], errors="coerce")
clean["decision_date_parsed"] = pd.to_datetime(clean["Decision Date"], errors="coerce")

assert clean.shape[0] == raw.shape[0], "Cleaning unexpectedly changed row count."
assert clean["zip5"].notna().all(), "Missing/invalid ZIP encountered; review before key construction."

## 6. Retailer entity resolution
We maintain **two identifiers** because an address is a place, not necessarily a business. Multiple businesses can occupy the same address, either concurrently or over time.

In [ ]:
clean["location_key"] = clean["street_address_normalized"] + "|" + clean["zip5"]
clean["retailer_entity_key"] = (
    clean["retailer_name_normalized"] + "|" + clean["location_key"]
)
assert clean["location_key"].notna().all()
assert clean["retailer_entity_key"].notna().all()

location_name_counts = clean.groupby("location_key")["retailer_name_normalized"].nunique()
ambiguous_locations = location_name_counts[location_name_counts > 1].sort_values(ascending=False)
clean["multi_name_location_flag"] = clean["location_key"].isin(ambiguous_locations.index).astype(int)

print("Unique normalized locations (address + ZIP):", clean["location_key"].nunique())
print("Conservative retailer entities (name + location):", clean["retailer_entity_key"].nunique())
print("Locations with >1 retailer name:", len(ambiguous_locations))
print("Maximum retailer names at one location:", int(location_name_counts.max()))

display(
    clean.loc[clean["multi_name_location_flag"].eq(1),
              ["Retailer Name", "Street Address", "Zip", "Decision Date"]]
         .drop_duplicates()
         .sort_values(["Street Address", "Decision Date"])
         .head(40)
)

**Interpretation:** A multi-name location can indicate a business rename, ownership turnover, multiple tenants at one address, or source inconsistency. We do not automatically merge these cases. `location_key` should later anchor geocoding/routing, while `retailer_entity_key` is the conservative history unit until a documented entity-resolution rule is adopted.

## 7. Outcome validation
We validate `Sale to UP` against related fields before constructing a binary historical outcome.

In [ ]:
print("UP Involved × Sale to UP")
display(pd.crosstab(clean["UP Involved"], clean["Sale to UP"], dropna=False, margins=True))
print("\nSale to UP × Outcome")
display(pd.crosstab(clean["Sale to UP"], clean["Outcome"], dropna=False, margins=True))

unexpected_sale = ~clean["Sale to UP"].isin(["Yes", "No"])
print("Unexpected/missing Sale to UP values:", int(unexpected_sale.sum()))
assert not unexpected_sale.any(), "Outcome coding is not purely Yes/No; inspect before binary recode."

clean["underage_sale_violation"] = clean["Sale to UP"].map({"Yes": 1, "No": 0}).astype("int8")
assert set(clean["underage_sale_violation"].unique()).issubset({0, 1})

outcome_summary = pd.Series({
    "positive_records": int(clean["underage_sale_violation"].sum()),
    "negative_records": int((clean["underage_sale_violation"] == 0).sum()),
    "record_level_positive_rate": float(clean["underage_sale_violation"].mean()),
})
display(outcome_summary.to_frame("value"))

**Outcome definition:** `underage_sale_violation = 1` when the FDA export explicitly records `Sale to UP == "Yes"`; otherwise it is 0 only for rows explicitly recorded `No`. This is a historical observed-inspection outcome—not evidence that unobserved retailers are compliant.

## 8. Exploratory analysis
These plots describe the observed FDA records. They should not be interpreted as population-level retailer prevalence estimates.

In [ ]:
# Decision year is used for broad temporal EDA because it is complete in this export.
clean["decision_year"] = clean["decision_date_parsed"].dt.year.astype("Int64")
yearly = clean.groupby("decision_year", dropna=False).agg(
    records=("retailer_entity_key", "size"),
    underage_sale_violations=("underage_sale_violation", "sum"),
    violation_rate=("underage_sale_violation", "mean"),
).reset_index()
display(yearly)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(yearly["decision_year"], yearly["records"], marker="o")
ax.set_title("FDA records by decision year")
ax.set_xlabel("Decision year")
ax.set_ylabel("Number of records")
ax.grid(alpha=.2)
plt.show()

**Interpretation:** Changes in annual record volume can reflect enforcement capacity, sampling strategy, reporting, or retailer activity. Later risk models should not assume the inspection process is stationary across years.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(yearly["decision_year"], yearly["violation_rate"], marker="o")
ax.set_title("Observed underage-sale violation rate by decision year")
ax.set_xlabel("Decision year")
ax.set_ylabel("Observed violation rate")
ax.set_ylim(bottom=0)
ax.grid(alpha=.2)
plt.show()

**Interpretation:** This is `P(recorded sale to UP | FDA record in that year)`, not the citywide prevalence of illegal sales. Year effects may capture both retailer behavior and changes in who was inspected.

In [ ]:
entity_counts = clean.groupby("retailer_entity_key").size()
entity_violations = clean.groupby("retailer_entity_key")["underage_sale_violation"].sum()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(entity_counts, bins=range(1, int(entity_counts.max()) + 2), align="left", rwidth=.85)
ax.set_title("Distribution of FDA records per conservative retailer entity")
ax.set_xlabel("Number of FDA records")
ax.set_ylabel("Retailer entities")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(entity_violations, bins=range(0, int(entity_violations.max()) + 2), align="left", rwidth=.85)
ax.set_title("Distribution of recorded underage-sale violations per retailer entity")
ax.set_xlabel("Recorded violations")
ax.set_ylabel("Retailer entities")
plt.show()

**Interpretation:** Repeated records create unequal historical exposure. Raw violation counts therefore mix risk with the number of opportunities a retailer had to be observed.

In [ ]:
print("Outcome categories")
display(clean["Outcome"].value_counts(dropna=False).to_frame("records"))
print("Product types")
display(clean["Product Type"].value_counts(dropna=False).head(20).to_frame("records"))

## 9. Temporal analysis
Because `Inspection Date` is sparse, intervals are computed using decision dates and explicitly labeled as **record/decision intervals**, not inspection intervals.

In [ ]:
tmp = clean.sort_values(["retailer_entity_key", "decision_date_parsed"]).copy()
tmp["days_since_previous_decision"] = tmp.groupby("retailer_entity_key")["decision_date_parsed"].diff().dt.days
intervals = tmp["days_since_previous_decision"].dropna()
print(intervals.describe())

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(intervals.clip(upper=intervals.quantile(.99)), bins=40)
ax.set_title("Days between consecutive FDA decision records per retailer (99th percentile clipped)")
ax.set_xlabel("Days")
ax.set_ylabel("Intervals")
plt.show()

**Modeling implication:** Time-varying prediction must construct features using only records strictly before the target decision/inspection. Full-history aggregates below are descriptive and must not be copied directly into a historical backtest.

## 10. Retailer-level feature engineering
This table is a **descriptive full-history snapshot** at the notebook cutoff. It is useful for data integration and current-state planning, but several fields would leak future information in retrospective prediction.

In [ ]:
AS_OF_DATE = clean["decision_date_parsed"].max()
ordered = clean.sort_values(["retailer_entity_key", "decision_date_parsed"])

retailer_master = (
    ordered.groupby("retailer_entity_key", as_index=False)
    .agg(
        retailer_name=("Retailer Name", "last"),
        retailer_name_normalized=("retailer_name_normalized", "last"),
        street_address=("Street Address", "last"),
        street_address_normalized=("street_address_normalized", "last"),
        city=("City", "last"),
        state=("State", "last"),
        zip5=("zip5", "last"),
        location_key=("location_key", "last"),
        multi_name_location_flag=("multi_name_location_flag", "max"),
        total_fda_records=("retailer_entity_key", "size"),
        inspections_with_underage_purchaser=("UP Involved", lambda s: int(s.eq("Yes").sum())),
        underage_sale_violations=("underage_sale_violation", "sum"),
        historical_violation_rate=("underage_sale_violation", "mean"),
        first_known_inspection_date=("inspection_date_parsed", "min"),
        most_recent_known_inspection_date=("inspection_date_parsed", "max"),
        first_decision_date=("decision_date_parsed", "min"),
        most_recent_decision_date=("decision_date_parsed", "max"),
        most_recent_outcome=("Outcome", "last"),
    )
)
retailer_master["ever_recorded_underage_sale_violation"] = (retailer_master["underage_sale_violations"] > 0).astype("int8")
retailer_master["repeat_violation_flag"] = (retailer_master["underage_sale_violations"] > 1).astype("int8")
retailer_master["days_since_most_recent_decision"] = (AS_OF_DATE - retailer_master["most_recent_decision_date"]).dt.days

assert len(retailer_master) == clean["retailer_entity_key"].nunique()
assert retailer_master["total_fda_records"].sum() == len(clean)
assert retailer_master["retailer_entity_key"].is_unique

display(retailer_master.head())
print("Retailer entities:", len(retailer_master))
print("Entities with >=1 recorded underage sale:", int(retailer_master["ever_recorded_underage_sale_violation"].sum()))

### Leakage warning
For a future prediction made at time **t**, features such as `total_fda_records`, `historical_violation_rate`, `ever_recorded_underage_sale_violation`, and `most_recent_outcome` must be recomputed using only records dated **before t**. The exported master is a current/full-history integration table, not a ready-made historical training matrix.

## 11. Modeling and selection-bias considerations
The FDA data describe retailers that entered the inspection/enforcement data-generating process. Therefore the directly observed quantity is approximately **P(violation | retailer observed/inspected)**. The policy quantity of interest is closer to **P(violation | retailer exists)**. These are not interchangeable when inspection selection is non-random.

The Pennsylvania active license list is therefore necessary to define the retailer universe/denominator. A licensed retailer absent from FDA records should be coded as **not observed in this FDA history**, not as a zero-violation/compliant retailer. Later modeling should consider selection mechanisms, temporal validation, and exploration/randomization in inspection allocation.

## 12. External-data join preparation
The master retains normalized address, ZIP, and `location_key` for geocoding. The following fields are placeholders only; no unavailable observations are fabricated.

In [ ]:
future_fields = {
    "pa_license_match": pd.NA,
    "pa_license_type": pd.NA,
    "pa_license_name": pd.NA,
    "latitude": np.nan,
    "longitude": np.nan,
    "census_tract_geoid": pd.NA,
    "acs_total_population": np.nan,
    "acs_youth_population": np.nan,
    "acs_youth_share": np.nan,
    "acs_median_household_income": np.nan,
    "acs_poverty_rate": np.nan,
    "nearest_public_school_name": pd.NA,
    "nearest_public_school_nces_id": pd.NA,
    "nearest_public_school_distance_m": np.nan,
    "schools_within_1000m": np.nan,
}
for col, default in future_fields.items():
    retailer_master[col] = default

retailer_master["synar_role"] = "Aggregate calibration/validation; no retailer-level join without supporting identifiers"

## 13. Audit against prior derived files and export processed datasets
The prior files are comparators, not inputs to the regenerated outputs. Differences are reported rather than silently reconciled.

In [ ]:
comparison = []
if PRIOR_CLEAN_FILE.exists():
    prior_clean = pd.read_csv(PRIOR_CLEAN_FILE, low_memory=False)
    comparison.append({"file": PRIOR_CLEAN_FILE.name, "rows": len(prior_clean), "columns": prior_clean.shape[1]})
if PRIOR_MASTER_FILE.exists():
    prior_master = pd.read_csv(PRIOR_MASTER_FILE, low_memory=False)
    comparison.append({"file": PRIOR_MASTER_FILE.name, "rows": len(prior_master), "columns": prior_master.shape[1]})
if comparison:
    display(pd.DataFrame(comparison))

print("Regenerated conservative retailer entities:", len(retailer_master))
print("Note: a lower prior-master row count can indicate more aggressive entity collapsing.")

In [ ]:
# Export date columns as ISO-like strings through pandas CSV serialization.
clean.to_csv(OUT_CLEAN, index=False)
retailer_master.to_csv(OUT_MASTER, index=False)
print("Wrote:", OUT_CLEAN)
print("Wrote:", OUT_MASTER)

## 14. Final data-quality report

In [ ]:
quality_metrics = [
    ("raw_observations", len(raw)),
    ("cleaned_observations", len(clean)),
    ("exact_duplicate_rows_removed", 0),
    ("unique_normalized_locations", clean["location_key"].nunique()),
    ("conservative_retailer_entities", clean["retailer_entity_key"].nunique()),
    ("multi_name_locations_flagged", int((location_name_counts > 1).sum())),
    ("missing_or_unparseable_inspection_dates", int(clean["inspection_date_parsed"].isna().sum())),
    ("missing_or_unparseable_decision_dates", int(clean["decision_date_parsed"].isna().sum())),
    ("positive_underage_sale_records", int(clean["underage_sale_violation"].sum())),
    ("negative_underage_sale_records", int((clean["underage_sale_violation"] == 0).sum())),
    ("retailer_entities_with_recorded_violation", int(retailer_master["ever_recorded_underage_sale_violation"].sum())),
]
quality_summary = pd.DataFrame(quality_metrics, columns=["metric", "value"])
quality_summary.to_csv(OUT_QUALITY, index=False)
display(quality_summary)

# Final reconciliation checks
assert len(clean) == len(raw)
assert clean["underage_sale_violation"].sum() + (clean["underage_sale_violation"] == 0).sum() == len(clean)
assert retailer_master["total_fda_records"].sum() == len(clean)
print("All final reconciliation assertions passed.")

## EDA Findings & Modeling Implications

1. **The raw file contains FDA records, not a clean retailer census.** Repeated observations occur, so inspection history and retailer/location tables must remain distinct.
2. **Location and retailer identity are different concepts.** Address + ZIP is appropriate for a geographic `location_key`, but multiple retailer names can share a location. The conservative `retailer_entity_key` therefore includes retailer name and location until ambiguous cases are resolved.
3. **Do not over-normalize addresses.** Aggressive suffix replacement can collapse source strings and reduce the apparent retailer/location count without evidence that the entities are identical.
4. **Inspection dates are sparse in this export.** Decision dates are more complete and are used only as an explicitly labeled temporal proxy. No date imputation is performed.
5. **`Sale to UP` supports a clear observed binary outcome** for these FDA records, but the observed positive rate is conditional on the FDA observation/inspection process.
6. **Historical exposure is unequal.** Retailers with more FDA records have had more opportunities to accumulate recorded violations, so raw violation counts should not be interpreted as pure risk.
7. **Selection bias is central.** The active Pennsylvania licensing file is needed to establish the retailer universe. Absence from FDA data must remain “unobserved,” not “compliant.”
8. **Future prediction must be time-aware.** Full-history features in the exported master are descriptive. Backtesting must reconstruct features using only information available before each target date.
9. **External joins should use geography deliberately.** License matching should use business/address information; ACS should follow geocoding to tract GEOID; NCES should use coordinates for distance/buffer features; Synar should remain aggregate calibration/validation unless retailer-level identifiers actually exist.
10. **Next notebook:** integrate the Pennsylvania license universe first, then geocode locations and add ACS/NCES features before fitting the retailer-risk model.